In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

In [ ]:
# -------------------------------------------------------
# 1. Load your data
# -------------------------------------------------------
df = pd.read_csv("data.csv")  # rename file
t = df["time"].values
omega = df["vel"].values
iq = df["iq"].values

In [ ]:
# -------------------------------------------------------
# 2. Smooth velocity & compute acceleration
# -------------------------------------------------------
# Window must be odd, choose based on sampling time
# Polynomial order = 3 works robustly for acceleration
omega_s = savgol_filter(omega, window_length=51, polyorder=3)

# Acceleration = derivative of velocity
# Use SG filter again to obtain smooth derivative
domega = np.gradient(omega_s, t)
acc = savgol_filter(domega, window_length=51, polyorder=3)



In [ ]:
# -------------------------------------------------------
# 3. Build regression model:
#    iq = (J/kt) * acc + (B/kt) * omega_s
# -------------------------------------------------------
A = np.vstack([acc, omega_s]).T
y = iq

theta, *_ = np.linalg.lstsq(A, y, rcond=None)
J_over_kt, B_over_kt = theta

print("Estimated parameters:")
print(f" J/k_t  = {J_over_kt:.5f}")
print(f" B/k_t  = {B_over_kt:.5f}")

In [ ]:
# -------------------------------------------------------
# 4. Plot measured vs. predicted torque
# -------------------------------------------------------
iq_pred = J_over_kt * acc + B_over_kt * omega_s

plt.figure(figsize=(12,5))
plt.plot(t, iq, label="measured iq")
plt.plot(t, iq_pred, '--', label="predicted iq (model)")
plt.legend()
plt.xlabel("Time (s)")
plt.ylabel("iq / torque command")
plt.title("Model Fit (Torque via iq)")
plt.grid()
plt.show()
